In [1]:
import pandas as pd

## Load and Inspect

In [24]:
uk_2y_raw = pd.read_csv("../data/raw/2y_gilt/v1_11-09-25/uk-2y-data-asof-110925.csv")

In [3]:
uk_2y_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12177 entries, 0 to 12176
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        12175 non-null  object 
 1   uk_2y       11793 non-null  float64
 2   Unnamed: 2  0 non-null      float64
 3   Unnamed: 3  0 non-null      float64
 4   Unnamed: 4  0 non-null      float64
dtypes: float64(4), object(1)
memory usage: 475.8+ KB


In [4]:
uk_2y_raw.head()

,date,uk_2y,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,01 Jan 25,NaN,NaN,NaN,NaN
1,02 Jan 25,4.20,NaN,NaN,NaN
2,03 Jan 25,4.22,NaN,NaN,NaN
3,06 Jan 25,4.24,NaN,NaN,NaN
4,07 Jan 25,4.26,NaN,NaN,NaN


## Cleaning

### Column Structuring

In [5]:
uk_2y_clean = uk_2y_raw.drop(columns=["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"])

### Data Type Conversions

In [6]:
uk_2y_clean.dtypes

date      object
uk_2y    float64
dtype: object

In [7]:
def parse_mixed_dates(series, formats):
    result = pd.Series(pd.NaT, index=series.index)
    for fmt in formats:
        mask = result.isna()
        parsed = pd.to_datetime(series[mask], format=fmt, errors="coerce")
        result[mask] = parsed
    
    if result[result.isna()].count() != series[series.isna()].count():
        raise ValueError("Failed to parse all dates")

    return result

dates = parse_mixed_dates(uk_2y_clean["date"], ["%d %b %y", "%d-%b-%y"])

In [8]:
uk_2y_clean["date"] = dates

In [9]:
uk_2y_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12177 entries, 0 to 12176
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    12175 non-null  datetime64[ns]
 1   uk_2y   11793 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 190.4 KB


### Handling Null Dates

In [10]:
uk_2y_clean[uk_2y_clean["date"].isna()]

,date,uk_2y
173,NaT,NaN
12176,NaT,NaN


In [11]:
# Remove rows with missing dates
uk_2y_clean = uk_2y_clean[uk_2y_clean["date"].notnull()]
uk_2y_clean = uk_2y_clean.reset_index(drop=True)

In [12]:
uk_2y_clean[uk_2y_clean["date"].isna()]

,date,uk_2y


### Indexing

In [13]:
uk_2y_clean = uk_2y_clean.set_index("date").sort_index()
uk_2y_clean.head()

,uk_2y
date,
1979-01-01,NaN
1979-01-02,11.96
1979-01-03,11.99
1979-01-04,11.94
1979-01-05,11.89


### Missing Timestamps

In [14]:
full_range = pd.date_range(start=uk_2y_clean.index.min(), end=uk_2y_clean.index.max(), freq='B')
missing_dates = full_range.difference(uk_2y_clean.index)

print("Missing dates:", missing_dates)

Missing dates: DatetimeIndex([], dtype='datetime64[ns]', freq='B')


### Handling Missing Yield Values

Q: How do we want to handle missing values in this time-series? We want to calculate the spread smoothly over time and if either series is missing data then this will cause gaps. Could ff? From a domain perspective, it may be safe enough to assume that if there is no data then the yield for that day is the same as the previous day.

In [15]:
uk_2y_clean[uk_2y_clean["uk_2y"].isna()]

,uk_2y
date,
1979-01-01,NaN
1979-04-13,NaN
1979-04-16,NaN
1979-05-07,NaN
1979-05-28,NaN
...,...
2025-04-18,NaN
2025-04-21,NaN
2025-05-05,NaN


In [16]:
uk_2y_clean[uk_2y_clean["uk_2y"].notna()].index.to_series().diff().value_counts()

date
1 days    9323
3 days    2200
4 days     164
5 days      84
2 days      21
Name: count, dtype: int64

Using forward-fill (ffill): Assumes the yield held steady until the next observation. Especially reasonable for short gaps since gilts don’t jump daily outside stress periods.

In [17]:
uk_2y_clean["uk_2y"] = uk_2y_clean["uk_2y"].ffill(limit=5)

In [18]:
uk_2y_clean[uk_2y_clean["uk_2y"].isna()]

,uk_2y
date,
1979-01-01,NaN


In [19]:
uk_2y_clean['uk_2y'] = uk_2y_clean['uk_2y'].bfill(limit=1)

In [20]:
uk_2y_clean[uk_2y_clean["uk_2y"].isna()]

,uk_2y
date,


### Time-Series Validation

In [22]:
uk_2y_clean.index[uk_2y_clean.index.duplicated()].shape[0] == 0

True

In [23]:
uk_2y_clean.index.is_monotonic_increasing

True

## Export

In [25]:
uk_2y_clean.to_csv("../data/cleaned/2y_gilt/v1_11-09-25/2y_gilt_cleaned_as_of_110925.csv")
uk_2y_clean.dtypes.to_csv("../data/cleaned/2y_gilt/v1_11-09-25/2y_gilt_cleaned_dtypes_as_of_110925.csv")